In [6]:
import random, datetime
from pathlib import Path
from collections import deque
import matplotlib.pyplot as plt

OUT_DIR = Path("./output"); OUT_DIR.mkdir(exist_ok=True)
NUM_RECORDS, ORDER, LEAF_CAP, OFFSET_START, OFFSET_STEP = 64, 4, 2, 22, 29

def gen_emp(outfile=OUT_DIR / "employee.txt"):
    random.seed(42)
    addr = ["TPE", "NTPC", "TXG", "TNN", "KHH"]
    s, e = datetime.date(1970, 1, 1), datetime.date(2000, 12, 31)
    def rb(): return (s + datetime.timedelta(days=random.randint(0, (e - s).days))).strftime("%Y/%m/%d")
    data = [(i, f"Employee_{i:02d}", rb(), random.choice(addr)) for i in range(1, NUM_RECORDS + 1)]
    with open(outfile, "w", encoding="utf-8") as f:
        f.write("id,name,birth,address\n")
        for d in data: f.write(",".join(map(str, d)) + "\n")
    return data

class Leaf:
    def __init__(self, cap): self.keys, self.values, self.next, self.cap = [], [], None, cap
class Internal:
    def __init__(self, order): self.keys, self.children, self.order = [], [], order

class BPlusTree:
    def __init__(self, order, cap):
        self.order, self.cap = order, cap
        self.root = Leaf(cap)

    def find_leaf(self, key, path=None):
        n = self.root
        if path is not None: path.append(n)
        while isinstance(n, Internal):
            i = sum(key >= k for k in n.keys)
            n = n.children[i]
            if path is not None: path.append(n)
        return n

    def insert(self, k, v):
        l = self.find_leaf(k)
        i = sum(x < k for x in l.keys)
        l.keys.insert(i, k); l.values.insert(i, v)
        if len(l.keys) > self.cap: self.split_leaf(l)

    def split_leaf(self, l):
        r = Leaf(self.cap); m = (len(l.keys)+1)//2
        r.keys, r.values, r.next = l.keys[m:], l.values[m:], l.next
        l.keys, l.values, l.next = l.keys[:m], l.values[:m], r
        if l is self.root:
            self.root = Internal(self.order)
            self.root.keys, self.root.children = [r.keys[0]], [l, r]
        else:
            p, _ = self.find_parent(self.root, l)
            self.insert_parent(p, r.keys[0], r)

    def find_parent(self, cur, tgt, path=None):
        if isinstance(cur, Internal):
            for c in cur.children:
                if c is tgt: return cur, path
                if isinstance(c, Internal):
                    r = self.find_parent(c, tgt, (path or []) + [cur])
                    if r: return r
        return None

    def insert_parent(self, p, key, rc):
        i = sum(k < key for k in p.keys)
        p.keys.insert(i, key); p.children.insert(i + 1, rc)
        if len(p.children) > self.order: self.split_internal(p)

    def split_internal(self, n):
        m = len(n.keys)//2; r = Internal(self.order); up = n.keys[m]
        r.keys, r.children = n.keys[m+1:], n.children[m+1:]
        n.keys, n.children = n.keys[:m], n.children[:m+1]
        if n is self.root:
            self.root = Internal(self.order)
            self.root.keys, self.root.children = [up], [n, r]
        else:
            p, _ = self.find_parent(self.root, n)
            self.insert_parent(p, up, r)

    def search(self, key):
        path = []
        l = self.find_leaf(key, path)
        for i, k in enumerate(l.keys):
            if k == key:
                return l.values[i], path
        return None, path

    def serialize(self):
        q, nodes, seen = deque([self.root]), [], set()
        while q:
            n = q.popleft()
            if id(n) in seen: continue
            seen.add(id(n)); nodes.append(n)
            if isinstance(n, Internal): q.extend(n.children)
        ids = {id(n): i for i, n in enumerate(nodes)}
        offs = {i: OFFSET_START + (i-1)*OFFSET_STEP for i in range(1, NUM_RECORDS+1)}
        lines = []
        for i, n in enumerate(nodes):
            if isinstance(n, Internal):
                lines.append(f"NODE {i} TYPE=I KEYS={n.keys} CHILDREN={[ids[id(c)] for c in n.children]}")
            else:
                ent = [f"{k}@{offs[k]}" for k in n.keys]
                nxt = ids.get(id(n.next), -1) if n.next else -1
                lines.append(f"NODE {i} TYPE=L KEYS={n.keys} ENTRIES={ent} NEXT={nxt}")
        return "\n".join(lines)

    def draw_tree(self, filename="tree.png", highlight_path=None):
        # -------- 構建節點層級 --------
        def get_levels():
            n = self.root
            while isinstance(n, Internal): n = n.children[0]
            leaves, cur = [], n
            while cur: leaves.append(cur); cur = cur.next
            lv, cur = [leaves], leaves
            while True:
                ps = []
                for c in cur:
                    p = self.find_parent(self.root, c)
                    if p and p[0] not in ps: ps.append(p[0])
                if not ps: break
                lv.append(ps); cur = ps
            return lv

        lvls = get_levels()
        nodes = [n for l in reversed(lvls) for n in l]
        idx = {id(n): i for i, n in enumerate(nodes)}
        child = {i:[idx[id(c)] for c in n.children] for i,n in enumerate(nodes) if isinstance(n,Internal)}
        leaves = [i for i,n in enumerate(nodes) if isinstance(n,Leaf)]

        # -------- 佈局座標 --------
        xgap = 2.5
        leaf_x = {i:j*xgap for j,i in enumerate(leaves)}
        pos = {i:(leaf_x[i],0) for i in leaves}
        def pos_calc(i):
            if i in pos: return pos[i]
            cpos = [pos_calc(c) for c in child[i]]
            x = sum(px for px,_ in cpos)/len(cpos); y = max(py for _,py in cpos)+1
            pos[i]=(x,y); return pos[i]
        pos_calc(0)

        # -------- 畫圖 --------
        fig, ax = plt.subplots(figsize=(14,6)); ax.axis("off")
        path_ids = {id(n) for n in highlight_path} if highlight_path else set()

        for p, cs in child.items():
            x1, y1 = pos[p]
            for c in cs:
                x2, y2 = pos[c]
                on_path = id(nodes[p]) in path_ids and id(nodes[c]) in path_ids
                ax.plot([x1,x2],[y1-0.3,y2+0.3],
                        lw=2.8 if on_path else 0.8, color="red" if on_path else "black")

        for i, n in enumerate(nodes):
            x, y = pos[i]
            highlight = id(n) in path_ids
            rect = plt.Rectangle((x-0.8,y-0.3),1.6,0.6,
                                 fc="wheat",
                                 ec="red" if highlight else "black",
                                 lw=2 if highlight else 1.0)
            ax.add_patch(rect)
            ax.text(x, y, ",".join(map(str, n.keys)),
                    ha="center", va="center",
                    fontsize=8, color="red" if highlight else "black")

        fig.savefig(OUT_DIR/filename,dpi=150)
        plt.close(fig)
        print(f"[+] {filename} generated")

# ---------------- main ----------------
if __name__ == "__main__":
    emps = gen_emp()
    tree = BPlusTree(ORDER, LEAF_CAP)
    for e in emps: tree.insert(e[0], e)

    with open(OUT_DIR/"BPlusTree.txt","w",encoding="utf-8") as f:
        f.write(tree.serialize())

    tree.draw_tree("tree.png")

    try: sid = int(input("請輸入要查詢的 id 值 (1~64): "))
    except: sid = 30

    res, path = tree.search(sid)
    if res:
        print(f"找到 id={sid} : {res}")
        # 🔴 顯示紅色搜尋路徑
        tree.draw_tree("search.png", highlight_path=path)
    else:
        print(f"未找到 id={sid}")


[+] tree.png generated
請輸入要查詢的 id 值 (1~64): 25
找到 id=25 : (25, 'Employee_25', '1975/08/08', 'TNN')
[+] search.png generated
